In [1]:
import torch

For $a=-m$, we have:

$$
\begin{align}
{}_2 F_1(-m,b,c,z) &= \sum_{k=0}^\infty \frac{(-m)_k (b)_k}{(c)_k} \frac{z^k}{k!} \\
&=  m! \sum_{k=0}^m \frac{(-1)^k(b)_k}{ (m-k-1)!(c)_k} \frac{z^k}{k!} 
\end{align}
$$

where $(x)_k = (x+k-1)!  / (x-1)!$

In [85]:
m = 16
a = -m
b = 1.

In [86]:
z = torch.rand(4,10).double().cuda()
z.requires_grad_(True)
z

tensor([[0.9055, 0.3673, 0.8773, 0.1364, 0.5405, 0.4177, 0.3340, 0.3158, 0.3760,
         0.7407],
        [0.0890, 0.7475, 0.5660, 0.2092, 0.7389, 0.0929, 0.2385, 0.4230, 0.0976,
         0.0594],
        [0.9093, 0.4455, 0.4978, 0.0342, 0.7415, 0.9984, 0.6669, 0.1074, 0.8014,
         0.7494],
        [0.1069, 0.0647, 0.8809, 0.7471, 0.0543, 0.0092, 0.2016, 0.8695, 0.1081,
         0.1180]], device='cuda:0', dtype=torch.float64, requires_grad=True)

In [87]:
c = 1.5 * torch.ones(4,1).double().cuda()

In [88]:
coef = 1.0
h2f1 = 1.0
for k in range(1, m+2):
    coef = coef  * (a+k-1) *(b+k-1) * z / (c+k-1) / k
    h2f1 += coef


In [89]:
h2f1

tensor([[0.0336, 0.0884, 0.0347, 0.2808, 0.0578, 0.0765, 0.0986, 0.1053, 0.0861,
         0.0414],
        [0.4156, 0.0410, 0.0550, 0.1726, 0.0415, 0.4016, 0.1474, 0.0754, 0.3855,
         0.5466],
        [0.0334, 0.0712, 0.0631, 0.7007, 0.0413, 0.0304, 0.0462, 0.3546, 0.0381,
         0.0409],
        [0.3560, 0.5197, 0.0346, 0.0410, 0.5745, 0.9075, 0.1804, 0.0350, 0.3525,
         0.3247]], device='cuda:0', dtype=torch.float64,
       grad_fn=<AddBackward0>)

In [90]:
h2f1.sum().backward()

In [91]:
g1 = z.grad.clone()

In [92]:
g1

tensor([[-0.0384, -0.2741, -0.0410, -2.1491, -0.1146, -0.2034, -0.3428, -0.3913,
         -0.2595, -0.0585],
        [-3.6967, -0.0574, -0.1037, -0.9920, -0.0588, -3.5334, -0.7485, -0.1977,
         -3.3465, -5.2411],
        [-0.0381, -0.1757, -0.1372, -7.0786, -0.0583, -0.0314, -0.0730, -2.9882,
         -0.0496, -0.0571],
        [-3.0046, -4.9223, -0.0407, -0.0574, -5.5729, -9.5567, -1.0700, -0.0418,
         -2.9648, -2.6457]], device='cuda:0', dtype=torch.float64)

In [93]:
z.grad.zero_()

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], device='cuda:0',
       dtype=torch.float64)

In [94]:
from pgcuts import Hyp2F1

In [95]:
t_h2f1 = Hyp2F1.apply(-m, b, c,z)

In [96]:
t_h2f1 - h2f1

tensor([[-1.1054e-14, -1.6653e-16,  1.7480e-13, -1.8319e-15, -5.5511e-16,
          4.3160e-15,  0.0000e+00,  9.2981e-16,  1.4155e-15, -7.0832e-14],
        [-2.6645e-15, -5.2874e-15,  4.8989e-15, -1.0825e-15,  1.8360e-14,
         -2.6090e-15, -9.1593e-16, -2.2621e-15, -2.4425e-15, -3.3307e-15],
        [ 1.5564e-14, -1.0686e-15,  1.4710e-15, -3.8858e-15,  3.0587e-14,
         -3.8858e-14,  1.2011e-14, -2.2204e-15, -5.1112e-14, -5.2340e-14],
        [-2.3870e-15, -3.3307e-15, -2.5384e-13,  1.9554e-14, -3.7748e-15,
         -5.6621e-15, -1.3600e-15, -1.2406e-13, -2.2204e-15, -2.0539e-15]],
       device='cuda:0', dtype=torch.float64, grad_fn=<SubBackward0>)

In [97]:
t_h2f1.sum().backward()
g2 = z.grad.clone()

In [98]:
g2

tensor([[-0.0384, -0.2741, -0.0410, -2.1491, -0.1146, -0.2034, -0.3428, -0.3913,
         -0.2595, -0.0585],
        [-3.6967, -0.0574, -0.1037, -0.9920, -0.0588, -3.5334, -0.7485, -0.1977,
         -3.3465, -5.2411],
        [-0.0381, -0.1757, -0.1372, -7.0786, -0.0583, -0.0314, -0.0730, -2.9882,
         -0.0496, -0.0571],
        [-3.0046, -4.9223, -0.0407, -0.0574, -5.5729, -9.5567, -1.0700, -0.0418,
         -2.9648, -2.6457]], device='cuda:0', dtype=torch.float64)

In [99]:
torch.allclose(g1,g2)

True

In [100]:
g1 - g2

tensor([[ 3.6934e-12, -9.9920e-15, -1.1591e-13, -8.8818e-15, -2.9088e-14,
          2.6368e-14,  2.1094e-15, -1.1713e-14,  1.6820e-14, -2.2268e-13],
        [-1.8208e-14, -3.8105e-13,  7.7785e-14, -6.6613e-16,  5.4277e-13,
         -1.8652e-14, -4.4409e-15,  3.6693e-14, -1.5099e-14, -2.3093e-14],
        [ 4.0912e-12,  1.6015e-14,  1.4211e-14, -3.1086e-14,  1.0034e-13,
         -2.7282e-12,  1.5350e-13, -1.1102e-14,  7.1086e-13,  4.3299e-15],
        [-1.2434e-14, -2.1316e-14,  1.6185e-12, -3.3375e-13, -2.6645e-14,
         -4.4409e-14, -9.7700e-15,  1.5296e-12, -1.3767e-14, -1.2879e-14]],
       device='cuda:0', dtype=torch.float64)

In [26]:
from scipy.special import hyp2f1

In [28]:
hyp2f1(a,b,c, z.cpu().numpy())

array([0.0478187 , 0.17316458, 0.03301376, 0.04363698, 0.04411342,
       0.04597748, 0.03074406, 0.03370566, 0.03596214, 0.04243905])